Shannon Hall

8/13/25

This program uses the algorithm outlined in section 3 of the paper [Block-Circulant Complex Hadamard Matrices](https://arxiv.org/abs/2204.11727) to find complex Hadamard matrices. Recall that a complex Hadamard matrix must satisfy two properties:
* All entries of the matrix have absolute value $1$
* The rows of the matrix are pairwise orthogonal

In a sense, the algorithm described in the paper alternates back and forth between these two properties in the hopes of converging to a Hadamard matrix. To go further, here we add extra steps to the algorithm in order to force the resulting Hadamard matrix to satisfy certain symmetries, such as forcing the matrix to be symmetric or forcing it to have certain specific entries.

# Functions used in the algorithm

In [12]:
from pandas import DataFrame
from scipy.linalg import polar
import numpy as np
import sympy as sp

# Set the size of the Hadamard matrix that we're looking for and the threshold for error
n = 7
threshold = 1e-13

# Get the nth Fourier matrix
def fourier(n):
    x = np.exp(2 * np.pi * 1j / n)
    return np.array([[x ** (i * j) for j in range(n)] for i in range(n)])

# Functions used to calculate the distance between two matrices
dist = lambda X, Y : np.linalg.norm(X - Y)
Z = lambda X : dist(X @ X.conj().T, n * np.eye(n))

# Normalize the entries of a matrix
normalize_entries = lambda X : X / np.abs(X)

# Get the unitary part from the polar decomposition of a matrix
polar_decomposition = lambda X : polar(X)[0]

# Get a dephased form of a Hadamard matrix
def dephase(X):
    D1 = np.diag([1 / row[0] for row in X])
    X = D1 @ X
    D2 = np.diag([1 / entry for entry in X[0]])
    return X @ D2

# Return the average between a matrix and its transpose
symmetric = lambda X : (X + X.T) / 2

# Replace the top-left corner of a matrix with another matrix
F = fourier(3)
def replace_corner(X, Y=F):
    len = Y.shape[0]
    A = np.copy(X)
    A[:len, :len] = Y
    return A

# Replace the top-left corner of a matrix with the average of the two matrices
def average_corner(X, Y=F):
    len = Y.shape[0]
    A = np.copy(X)
    average = (X[:len, :len] + Y) / 2
    A[:len, :len] = average
    return A

# Replace some entries of a matrix with their average
default_entries = [(1, 4), (2, 5)]
def average_entries_together(X, entries=default_entries):
    A = np.copy(X)
    average = np.mean([X[entry] for entry in entries])
    for entry in entries:
        A[entry] = average
    return A

# Replace some entries of a matrix with ones
default_entries = [(1, 1), (3, 3)]
def replace_entries_with_ones(X, entries=default_entries):
    A = np.copy(X)
    for entry in entries:
        A[entry] = 1
    return A

# Replace diagonal of a matrix with ones
def replace_diagonal_with_ones(X):
    return replace_entries_with_ones(X, entries=[(i, i) for i in range(n)])

# Create Petrescu's matrix
x = np.exp(2 * np.pi  * 1j / 6)
exponents = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 4, 5, 3, 3, 1],
    [0, 4, 1, 3, 5, 3, 1],
    [0, 5, 3, 1, 4, 1, 3],
    [0, 3, 5, 4, 1, 1, 3],
    [0, 3, 3, 1, 1, 4, 5],
    [0, 1, 1, 3, 3, 5, 4]])
P = np.power(x, exponents)

# Replace some entries of a matrix with entries from another matrix
default_entries = [(1, 3), (1, 4), (1, 5)]
def replace_entries(X, Y=P, entries=default_entries):
    A = np.copy(X)
    for entry in entries:
        A[entry] = Y[entry]
    return A

# Replace some entries in the first row
def special_replace(X):
    A = np.copy(X)
    A[1, 1] = 1
    A[1, 2] = -1
    A[1, 3] = 1j
    A[1, 4] = -1j
    return A

# Other useful functions

In [13]:
# Check that the given (square) matrix is Hadamard
def is_hadamard(X, threshold=threshold):
    return np.allclose(np.abs(X), 1) and Z(X) < threshold

# These functions are used to print matrices nicely
np.set_printoptions(precision=3, suppress=True)
roundentries = lambda entry : sp.re(entry).round(3) + sp.im(entry).round(3) * sp.I
quickdisplay = lambda M : display(sp.Matrix(M).applyfunc(roundentries))

# The algorithm itself

In [14]:
# Get a random matrix with entries on the unit circle
X = np.exp(2 * np.pi * 1j * np.random.rand(n, n))

# Perform changes to X until it is close to a Hadamard matrix
iterations = 0
previous_distance = Z(X)
while (Z(X) > threshold):
    iterations += 1

    # Perform changes to X
    X = polar_decomposition(X)
    X = replace_corner(X)
    # X = symmetric(X)
    # X = average_entries_together(X, entries=[(1, 4), (2, 5)])
    # X = average_entries_together(X, entries=[(2, 4), (1, 5)])
    # X = replace_entries(X)
    X = normalize_entries(X)
    X = dephase(X)

    # Check if the distance is not changing
    if (abs(Z(X) - previous_distance) < threshold / 10000):
        print('Z(X) seems to not be changing anymore')
        break
    previous_distance = Z(X)

    # Print out the distance every 1000 iterations
    if (iterations % 1000 == 0):
        print(f'Iteration {iterations}, Z(X): {Z(X)}')
print(f'Finished in {iterations} iterations\n')

# Print out the Hadamard matrix and its distance from the identity
print('Potential Hadamard matrix:')
quickdisplay(X)
print(f'Z(X): {Z(X)}')

Iteration 1000, Z(X): 3.207617143997946e-05
Iteration 2000, Z(X): 2.424972045670378e-13
Z(X) seems to not be changing anymore
Finished in 2024 iterations

Potential Hadamard matrix:


Matrix([
[1.0,              1.0,              1.0,            1.0,              1.0,            1.0,              1.0],
[1.0,   -0.5 + 0.866*I,   -0.5 - 0.866*I,  0.5 + 0.866*I,  0.947 - 0.322*I, -0.5 - 0.866*I, -0.947 + 0.322*I],
[1.0,   -0.5 - 0.866*I,   -0.5 + 0.866*I,  0.5 + 0.866*I, -0.947 + 0.322*I, -0.5 - 0.866*I,  0.947 - 0.322*I],
[1.0,  0.752 - 0.659*I, -0.752 + 0.659*I,           -1.0,              1.0, -0.5 + 0.866*I,   -0.5 - 0.866*I],
[1.0,    0.5 + 0.866*I,    0.5 + 0.866*I, -0.5 - 0.866*I,             -1.0,  0.5 - 0.866*I,             -1.0],
[1.0,   -0.5 - 0.866*I,   -0.5 - 0.866*I,  0.5 - 0.866*I,   -0.5 + 0.866*I,  0.5 + 0.866*I,   -0.5 + 0.866*I],
[1.0, -0.752 + 0.659*I,  0.752 - 0.659*I,           -1.0,   -0.5 - 0.866*I, -0.5 + 0.866*I,              1.0]])

Z(X): 2.5447816079004884e-13


## Checking the result

In [15]:
# Get the dephased form of the Hadamard matrix and print it out
H = dephase(X)
quickdisplay(H)
print('Check that each entry is on the unit circle:')
quickdisplay(np.abs(H))
print('Check that the rows are orthogonal:')
quickdisplay(H @ H.conj().T)
print(f'Z(H): {Z(H)}')

Matrix([
[1.0,              1.0,              1.0,            1.0,              1.0,            1.0,              1.0],
[1.0,   -0.5 + 0.866*I,   -0.5 - 0.866*I,  0.5 + 0.866*I,  0.947 - 0.322*I, -0.5 - 0.866*I, -0.947 + 0.322*I],
[1.0,   -0.5 - 0.866*I,   -0.5 + 0.866*I,  0.5 + 0.866*I, -0.947 + 0.322*I, -0.5 - 0.866*I,  0.947 - 0.322*I],
[1.0,  0.752 - 0.659*I, -0.752 + 0.659*I,           -1.0,              1.0, -0.5 + 0.866*I,   -0.5 - 0.866*I],
[1.0,    0.5 + 0.866*I,    0.5 + 0.866*I, -0.5 - 0.866*I,             -1.0,  0.5 - 0.866*I,             -1.0],
[1.0,   -0.5 - 0.866*I,   -0.5 - 0.866*I,  0.5 - 0.866*I,   -0.5 + 0.866*I,  0.5 + 0.866*I,   -0.5 + 0.866*I],
[1.0, -0.752 + 0.659*I,  0.752 - 0.659*I,           -1.0,   -0.5 - 0.866*I, -0.5 + 0.866*I,              1.0]])

Check that each entry is on the unit circle:


Matrix([
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]])

Check that the rows are orthogonal:


Matrix([
[7.0,   0,   0,   0,   0,   0,   0],
[  0, 7.0,   0,   0,   0,   0,   0],
[  0,   0, 7.0,   0,   0,   0,   0],
[  0,   0,   0, 7.0,   0,   0,   0],
[  0,   0,   0,   0, 7.0,   0,   0],
[  0,   0,   0,   0,   0, 7.0,   0],
[  0,   0,   0,   0,   0,   0, 7.0]])

Z(H): 2.5447816079004884e-13


## Finding a Butson-type matrix that is close to the matrix we found

In [16]:
# Compute the mth root of unity and its powers
m = 6
x = np.exp(2 * np.pi * 1j / m)
roots = [x ** k for k in range(m)]
print('Roots of unity:')
for k in range(m):
    print(f'x^{k} = {x ** k:.3f}')
print()

# Find the root of unity that is closest to the given entry
def closest_root(entry):
    closest = 0
    closest_dist = np.abs(entry - roots[closest])
    for k in range(1, m):
        dist = np.abs(entry - roots[k])
        if dist < closest_dist:
            closest = k
            closest_dist = dist
    return closest

# Find a Butson matrix that is close to H
powers_matrix = [[closest_root(entry) for entry in row] for row in H]
butson_matrix = np.power(x, powers_matrix)
print('Powers:')
quickdisplay(powers_matrix)
print('Original Hadamard matrix:')
quickdisplay(H)
print('Butson matrix:')
quickdisplay(butson_matrix)
print('Distance between entries:')
quickdisplay(np.abs(butson_matrix - H))

print(f'Z(H)      : {Z(H)}')
print(f'Z(B)      : {Z(butson_matrix)}')
print(f'dist(B, H): {dist(butson_matrix, H)}')

Roots of unity:
x^0 = 1.000+0.000j
x^1 = 0.500+0.866j
x^2 = -0.500+0.866j
x^3 = -1.000+0.000j
x^4 = -0.500-0.866j
x^5 = 0.500-0.866j

Powers:


Matrix([
[0, 0, 0, 0, 0, 0, 0],
[0, 2, 4, 1, 0, 4, 3],
[0, 4, 2, 1, 3, 4, 0],
[0, 5, 2, 3, 0, 2, 4],
[0, 1, 1, 4, 3, 5, 3],
[0, 4, 4, 5, 2, 1, 2],
[0, 2, 5, 3, 4, 2, 0]])

Original Hadamard matrix:


Matrix([
[1.0,              1.0,              1.0,            1.0,              1.0,            1.0,              1.0],
[1.0,   -0.5 + 0.866*I,   -0.5 - 0.866*I,  0.5 + 0.866*I,  0.947 - 0.322*I, -0.5 - 0.866*I, -0.947 + 0.322*I],
[1.0,   -0.5 - 0.866*I,   -0.5 + 0.866*I,  0.5 + 0.866*I, -0.947 + 0.322*I, -0.5 - 0.866*I,  0.947 - 0.322*I],
[1.0,  0.752 - 0.659*I, -0.752 + 0.659*I,           -1.0,              1.0, -0.5 + 0.866*I,   -0.5 - 0.866*I],
[1.0,    0.5 + 0.866*I,    0.5 + 0.866*I, -0.5 - 0.866*I,             -1.0,  0.5 - 0.866*I,             -1.0],
[1.0,   -0.5 - 0.866*I,   -0.5 - 0.866*I,  0.5 - 0.866*I,   -0.5 + 0.866*I,  0.5 + 0.866*I,   -0.5 + 0.866*I],
[1.0, -0.752 + 0.659*I,  0.752 - 0.659*I,           -1.0,   -0.5 - 0.866*I, -0.5 + 0.866*I,              1.0]])

Butson matrix:


Matrix([
[1.0,            1.0,            1.0,            1.0,            1.0,            1.0,            1.0],
[1.0, -0.5 + 0.866*I, -0.5 - 0.866*I,  0.5 + 0.866*I,            1.0, -0.5 - 0.866*I,           -1.0],
[1.0, -0.5 - 0.866*I, -0.5 + 0.866*I,  0.5 + 0.866*I,           -1.0, -0.5 - 0.866*I,            1.0],
[1.0,  0.5 - 0.866*I, -0.5 + 0.866*I,           -1.0,            1.0, -0.5 + 0.866*I, -0.5 - 0.866*I],
[1.0,  0.5 + 0.866*I,  0.5 + 0.866*I, -0.5 - 0.866*I,           -1.0,  0.5 - 0.866*I,           -1.0],
[1.0, -0.5 - 0.866*I, -0.5 - 0.866*I,  0.5 - 0.866*I, -0.5 + 0.866*I,  0.5 + 0.866*I, -0.5 + 0.866*I],
[1.0, -0.5 + 0.866*I,  0.5 - 0.866*I,           -1.0, -0.5 - 0.866*I, -0.5 + 0.866*I,            1.0]])

Distance between entries:


Matrix([
[0,     0,     0, 0,     0, 0,     0],
[0,     0,     0, 0, 0.326, 0, 0.326],
[0,     0,     0, 0, 0.326, 0, 0.326],
[0, 0.326, 0.326, 0,     0, 0,     0],
[0,     0,     0, 0,     0, 0,     0],
[0,     0,     0, 0,     0, 0,     0],
[0, 0.326, 0.326, 0,     0, 0,     0]])

Z(H)      : 2.5447816079004884e-13
Z(B)      : 6.638570889487459e-15
dist(B, H): 0.9217117871431048
